# Fabric Defect Classification — Data Preprocessing

**Notebook 02** of the fabric defect classification pipeline.

Notebook 01 explored the raw data and found several problems. This notebook fixes them and produces a clean dataset ready for training.

**What Notebook 01 told us:**

- The download contains 3,067 files, but only **2,758 are genuinely different images**.
- Three classes contain grayscale **processed copies** made from originals. These add no new information, and because they exist in only three classes they give the model a shortcut.
- There are **17 exact duplicates** saved under different names, and **40 images** in near-identical clusters caused by file copying.
- Classes are **severely imbalanced** — "defect free" is 60.4% of the data, while "Vertical" and "horizontal" are about 1.2% each. The largest class is 52 times the smallest.
- Image sizes vary hugely, and files come in a mix of JPEG and PNG, colour and grayscale.
- **No fabric roll or batch information exists**, so we cannot split the data by production batch.

**What this notebook does:**

1. Removes every kind of duplicate
2. Checks whether image size gives away the class
3. Converts all images to one consistent colour format and size
4. Splits the data using cross-validation
5. Calculates weights to correct the class imbalance
6. Saves everything for Notebook 03

## 1. Setup

**What:** Load the libraries and locate the dataset.

**Why:** Each notebook runs on its own, so we reload the same dataset that Notebook 01 used. `OUT_DIR` is where the cleaned images will be written — change it if you are not running on Kaggle.

In [ ]:
import hashlib
import json
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold

# ---------------- Configuration ----------------
IMG_SIZE = 224
N_FOLDS = 5
SEED = 42

# Where to write the cleaned data.
# Leave as None to choose automatically, or set it yourself, e.g.
#   OUT_DIR = Path("/kaggle/working/processed")   # Kaggle
#   OUT_DIR = Path("data/processed")              # local
OUT_DIR = None
# -----------------------------------------------

random.seed(SEED)

# Locate the raw dataset
ds2 = "/kaggle/input/multi-class-fabric-defect-detection-dataset/Dataset"
if not os.path.exists(ds2):
    import kagglehub
    ds2 = os.path.join(
        kagglehub.dataset_download("ziya07/multi-class-fabric-defect-detection-dataset"),
        "Dataset",
    )

# Resolve the output location if it was left automatic
if OUT_DIR is None:
    OUT_DIR = Path("/kaggle/working/processed") if Path("/kaggle/working").exists() \
        else Path("data/processed")

class_names = sorted(os.listdir(ds2))

print("Raw dataset :", ds2)
print("Output      :", OUT_DIR.resolve())
print("Classes     :", class_names)

## 2. Remove the Duplicates

**What:** Build the final list of images to keep, removing all three kinds of duplicate found in Notebook 01.

**Why:** This is the most important cleaning step. If two files trace back to the same original photograph and end up on opposite sides of a train/test split, the model is being graded on a picture it has already memorised — the score would look excellent and mean nothing.

Notebook 01 found three separate kinds of duplicate, and each needs its own detection method:

| Kind | What it is | How we detect it |
|---|---|---|
| **Processed derivative** | A grayscale version made from an original | The `_processed` ending in the filename |
| **Exact duplicate** | The identical file under a different name | An MD5 fingerprint of the file's bytes |
| **Near-duplicate** | The same picture re-saved, so the bytes differ | A difference hash comparing how the image looks |

We apply them in that order, keeping the first image we meet from each group and dropping the rest.

In [ ]:
def dhash(path, size=16):
    """Fingerprint based on appearance: compares each pixel with its right neighbour."""
    with Image.open(path) as img:
        small = img.convert("L").resize((size + 1, size), Image.BILINEAR)
    px = list(small.getdata())
    bits = []
    for row in range(size):
        for col in range(size):
            left = px[row * (size + 1) + col]
            right = px[row * (size + 1) + col + 1]
            bits.append("1" if left > right else "0")
    return "".join(bits)


# --- Step 1: drop the processed derivatives -------------------------------
candidates = [
    (cls, p)
    for cls in class_names
    for p in sorted((Path(ds2) / cls).glob("*"))
    if "_processed" not in p.name
]
raw_total = sum(len(list((Path(ds2) / cls).glob("*"))) for cls in class_names)
print(f"Raw files                     : {raw_total}")
print(f"After removing _processed     : {len(candidates)}")

# --- Step 2: drop exact duplicates ---------------------------------------
seen_md5, after_exact = set(), []
for cls, p in candidates:
    h = hashlib.md5(p.read_bytes()).hexdigest()
    if h in seen_md5:
        continue
    seen_md5.add(h)
    after_exact.append((cls, p))
print(f"After removing exact copies   : {len(after_exact)}")

# --- Step 3: drop near-duplicates (within each class) --------------------
seen_dhash, keep = set(), []
for cls, p in after_exact:
    key = (cls, dhash(p))
    if key in seen_dhash:
        continue
    seen_dhash.add(key)
    keep.append((cls, p))
print(f"After removing near-duplicates: {len(keep)}")
print(f"\nRemoved in total: {raw_total - len(keep)} files")

**Observations:** _(fill in — how many images were removed at each stage, and the final count)_

In [ ]:
final_counts = Counter(cls for cls, _ in keep)
total = len(keep)

print(f"{'Class':20s} {'Images':>7s} {'Share':>8s}")
for cls, n in sorted(final_counts.items(), key=lambda x: -x[1]):
    print(f"{cls:20s} {n:7d} {n/total:7.1%}")

print(f"\nTotal: {total}")
print(f"Imbalance ratio: {max(final_counts.values()) / min(final_counts.values()):.0f}:1")

**Observations:** _(fill in — the cleaned class counts, and whether the imbalance changed)_

## 3. Does Image Size Give Away the Class?

**What:** List the image sizes and colour formats used by each class.

**Why:** This decides how we resize, and it matters more than it sounds.

Notebook 01 found only 13 different image sizes across the whole dataset, and that the data came from several separate collections. If each collection used its own camera, then each class may have its own characteristic image size — and size would quietly become a clue to the answer.

That would rule out one popular resizing method. **Letterboxing** shrinks an image to fit and fills the leftover space with black bars, which keeps the original proportions intact. But if proportions differ by class, those black bars would tell the model which class it is looking at, without it ever examining the fabric.

So we check first, then choose.

In [ ]:
print(f"{'Class':16s} {'Colour formats':26s} {'Most common sizes'}")
print("-" * 90)
for cls in class_names:
    modes, sizes = Counter(), Counter()
    for c, p in keep:
        if c != cls:
            continue
        with Image.open(p) as img:
            modes[img.mode] += 1
            sizes[img.size] += 1
    top = ", ".join(f"{w}x{h} ({n})" for (w, h), n in sizes.most_common(3))
    print(f"{cls:16s} {str(dict(modes)):26s} {top}")

**Observations:** _(fill in — does each class have its own image size, or do sizes appear across several classes? Which colour formats are present?)_

**Decision:** We resize every image **directly to 224×224**, accepting that proportions get squashed, rather than letterboxing.

Two reasons:

1. **It removes the size clue.** Every image ends up the same shape, so nothing about the original camera survives into the model's input.
2. **Squashing does not harm the orientation defects.** "Vertical", "horizontal" and "lines" are defined by *direction*, and stretching an image does not rotate anything — a vertical line stays vertical. We lose true proportions, but we keep the feature that actually identifies the defect.

224×224 is the input size ResNet18 was originally trained on, so its pretrained weights transfer cleanly.

## 4. Standardise Every Image

**What:** Convert each image to colour format and resize it to 224×224.

**Why:** Notebook 01 found the images arrive in inconsistent forms — some JPEG and some PNG, some full colour and some grayscale, in 13 different sizes. A neural network needs every input to be identical in shape and channel count.

Converting to `RGB` handles all of it at once: grayscale images get their single brightness value copied into three colour channels, and PNG transparency is discarded. Every image ends up as 224×224 with three channels, regardless of how it started.

In [ ]:
def preprocess(path, size=IMG_SIZE):
    """Convert to 3-channel colour and resize to a fixed square."""
    with Image.open(path) as img:
        return img.convert("RGB").resize((size, size), Image.BILINEAR)


# Visual check: a few images before and after
samples = random.sample(keep, 4)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, (cls, p) in enumerate(samples):
    with Image.open(p) as img:
        axes[0, col].imshow(img.convert("RGB"))
        axes[0, col].set_title(f"{cls}\n{img.size[0]}x{img.size[1]}", fontsize=9)
    axes[1, col].imshow(preprocess(p))
    axes[1, col].set_title(f"{IMG_SIZE}x{IMG_SIZE}", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("original")
axes[1, 0].set_ylabel("processed")
plt.tight_layout()
plt.show()

**Observations:** _(fill in — are the defects still clearly visible after resizing? Did any image get distorted badly enough to be a problem?)_

## 5. Split the Data with Cross-Validation

**What:** Divide the images into 5 folds, keeping each class's proportions the same in every fold.

**Why:** The obvious approach would be a single split — say 70% train, 15% validation, 15% test. That fails here because of how small the rare classes are.

"Vertical" has about 32 images. A 15% test share would give it roughly **5 test images**. Each one would then be worth 20 percentage points of recall, so the score would jump around wildly depending on which 5 images happened to land there. That is not a measurement.

**Cross-validation** solves this. We split into 5 equal folds and train 5 times, each time holding out a different fold for testing. Every image gets tested exactly once, so "Vertical" ends up with all 32 predictions instead of 5. This directly supports reporting per-class recall for the rare defects.

We use `StratifiedKFold`, which keeps each class's share the same in every fold — without it, a fold could easily end up with no "Vertical" images at all.

**A note on grouping:** normally we would also make sure related images (same fabric roll, same photo session) stay together in one fold. Notebook 01 searched for that grouping twice — in the filenames, and by finding visually similar images — and found none. Since every duplicate has now been removed, each remaining image is independent, so a plain stratified split is correct here.

In [ ]:
paths = [p for _, p in keep]
labels = [cls for cls, _ in keep]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = {}
for fold_idx, (_, test_idx) in enumerate(skf.split(paths, labels)):
    for i in test_idx:
        fold_of[i] = fold_idx

print(f"{'Class':16s} " + " ".join(f"{'fold'+str(f):>7s}" for f in range(N_FOLDS)) + f" {'total':>7s}")
print("-" * 70)
for cls in class_names:
    row = [sum(1 for i, l in enumerate(labels) if l == cls and fold_of[i] == f)
           for f in range(N_FOLDS)]
    print(f"{cls:16s} " + " ".join(f"{n:7d}" for n in row) + f" {sum(row):7d}")

**Observations:** _(fill in — does every class appear in every fold? How many test images does "Vertical" get in total compared with a single 15% split?)_

## 6. Weight the Classes

**What:** Give each class a weight based on how rare it is.

**Why:** With "defect free" making up 60% of the data and "Vertical" only 1.2%, a model can score 60% accuracy by always answering "defect free" — Notebook 01 measured exactly this. Left uncorrected, training pushes the model straight towards that useless shortcut, because ignoring the rare classes barely affects the overall error.

Class weights fix this by changing the cost of a mistake. Getting a "Vertical" image wrong becomes far more expensive than getting a "defect free" image wrong, so the model can no longer afford to ignore the rare defects.

We use inverse frequency: rare classes receive large weights, common classes small ones.

**Why weights rather than duplicating images?** We could instead copy the rare images until every class was the same size. But with only 32 real "Vertical" images, that means showing the model the same 32 pictures over and over, and it would simply memorise them. Weighting avoids this. Genuine variety comes from augmentation during training, applied evenly to every class.

In [ ]:
# Recompute here so this cell works even if run on its own
final_counts = Counter(cls for cls, _ in keep)
total = len(keep)

class_weights = {
    cls: total / (len(class_names) * final_counts[cls])
    for cls in class_names
}

print(f"{'Class':20s} {'Images':>7s} {'Weight':>8s}")
for cls, w in sorted(class_weights.items(), key=lambda x: -x[1]):
    print(f"{cls:20s} {final_counts[cls]:7d} {w:8.2f}")

**Observations:** _(fill in — which classes get the heaviest weights, and roughly how much more than the common ones?)_

## 7. Save the Cleaned Dataset

**What:** Write every processed image to disk, along with a manifest listing each image's class and fold, and the class weights.

**Why:** Notebook 03 should be able to start training immediately, without repeating any of this cleaning work. Saving the fold assignments also makes the experiment **reproducible** — anyone re-running the project gets the exact same splits, so results can be compared fairly.

We store the fold in a manifest file rather than in folder names, because with cross-validation each image takes a turn as test data. One flat set of images plus a lookup table is simpler than five copies of everything.

In [ ]:
img_dir = OUT_DIR / "images"
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
img_dir.mkdir(parents=True, exist_ok=True)

rows = []
for i, (cls, p) in enumerate(keep):
    cls_dir = img_dir / cls
    cls_dir.mkdir(exist_ok=True)
    out_name = f"{i:05d}.jpg"
    preprocess(p).save(cls_dir / out_name, quality=95)
    rows.append({
        "filepath": str(Path("images") / cls / out_name),
        "class": cls,
        "fold": fold_of[i],
        "source_file": p.name,
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)

with open(OUT_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)

print(f"Saved {len(manifest)} images to {img_dir}")
print(f"Manifest     : {OUT_DIR / 'manifest.csv'}")
print(f"Class weights: {OUT_DIR / 'class_weights.json'}")
manifest.head()

## 8. Final Check

**What:** Reload what we just saved and confirm it is correct.

**Why:** Saving thousands of files is easy to get subtly wrong — a missing folder, an image written at the wrong size, a fold that lost its rare class. Checking now costs seconds; discovering it during training costs hours. We verify the counts match, every image really is 224×224 in colour format, and every class appears in every fold.

In [ ]:
check = pd.read_csv(OUT_DIR / "manifest.csv")
print(f"Rows in manifest: {len(check)}  (expected {len(keep)})")

# Every file exists, and has the right size and colour format
bad = []
for _, r in check.iterrows():
    fp = OUT_DIR / r["filepath"]
    if not fp.exists():
        bad.append((r["filepath"], "missing"))
        continue
    with Image.open(fp) as img:
        if img.size != (IMG_SIZE, IMG_SIZE) or img.mode != "RGB":
            bad.append((r["filepath"], f"{img.size} {img.mode}"))

print(f"Problem files: {len(bad)}")
for b in bad[:10]:
    print("  ", b)

print("\nImages per class per fold:")
print(pd.crosstab(check["class"], check["fold"], margins=True))

**Observations:** _(fill in — do the counts match, are all images the right size and format, and does every class appear in every fold?)_

## 9. Summary

### What this notebook did

1. **Removed duplicates** in three passes — processed derivatives by filename, exact copies by file fingerprint, and near-identical images by appearance.
2. **Checked whether image size reveals the class**, and chose direct square resizing over letterboxing so that no clue about the original camera reaches the model.
3. **Standardised every image** to 224×224 in three-channel colour, removing the JPEG/PNG and grayscale/colour inconsistencies found in Notebook 01.
4. **Split the data into 5 stratified folds**, so every image is tested exactly once and the rare classes get enough test predictions to measure.
5. **Calculated class weights** to stop the model defaulting to the majority class.
6. **Saved everything** — processed images, a manifest with fold assignments, and the class weights.

### Files produced

| File | Contents |
|---|---|
| `images/{class}/*.jpg` | Cleaned images, 224×224, colour |
| `manifest.csv` | Each image's class, fold, and original filename |
| `class_weights.json` | Weight per class for the training loss |

### Carrying this into Notebook 03

These files are written to `OUT_DIR`, which on Kaggle is inside the temporary session folder — **it is deleted when the session ends.** To make it available to Notebook 03:

1. In this notebook, click **Save Version → Save & Run All (Commit)**. This re-runs everything in a fresh session and keeps the output permanently.
2. From the finished version's output page, click **New Dataset** and name it (for example `fabric-defect-processed`).
3. In Notebook 03, click **+ Add Input** and attach that dataset.

Publishing it as a dataset rather than reading the notebook output directly keeps the path stable, so editing this notebook later cannot break Notebook 03.

If you are running locally instead, the files are already in `data/processed/` and Notebook 03 will find them there. Note that `data/` is excluded from git, so the processed images never reach GitHub — anyone cloning the repository re-runs this notebook to regenerate them.

### Decisions worth defending

- **Deleted the processed derivatives rather than using them as extra data.** They are grayscale versions of pictures we already have, and they exist in only three of the nine classes — so keeping them would let the model learn "grayscale means one of these three classes" instead of learning what the defects look like.
- **Squashed images to a square instead of padding them.** Padding preserves proportions, but proportions differ by source collection here, so the padding itself would leak the answer. Squashing loses proportion but keeps direction, which is what the orientation classes depend on.
- **Weighted the classes instead of copying rare images.** With only 32 real "Vertical" images, duplicating them would teach the model to memorise those 32 pictures.
- **Used cross-validation instead of one fixed split.** A single 15% test split would leave "Vertical" with about 5 test images, far too few for a per-class recall figure to mean anything.

### Still true from Notebook 01

- "Vertical" and "horizontal" have only around 32 and 34 images each. Their individual scores will carry real uncertainty no matter how well the model performs, and this should be stated alongside any result.
- No fabric roll or batch information exists, so splitting by production batch remains impossible.

### Next: Notebook 03

Train a simple CNN as a baseline, then ResNet18 using transfer learning, comparing both against the do-nothing baseline from Notebook 01 (60.4% accuracy, 0.084 macro F1).